In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import time
from zoneinfo import ZoneInfo

import databento as db
from arch import arch_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


In [3]:
client = db.Historical("")


In [4]:
dataset = "DBEQ.BASIC"
schema = "mbp-10"

start_time = "2025-08-02T09:30:00"
end_time   = "2025-09-02T16:00:00"
symbols = ["APA", "AR", "EQT", "RRC"]


In [5]:
NY = "America/New_York"

start_ny = pd.Timestamp("2025-08-02T09:30:00", tz=NY)
end_ny   = pd.Timestamp("2025-09-02T16:00:00", tz=NY)

start_api = start_ny.tz_convert("UTC").isoformat()
end_api   = end_ny.tz_convert("UTC").isoformat()

In [6]:
symbols = ["APA"]

In [8]:
import numpy as np
import pandas as pd
from datetime import time

# --- Fetch ---
bars = client.timeseries.get_range(
    dataset=dataset,
    schema=schema,
    start=start_api,   
    end=end_api,       
    symbols=symbols,
    stype_in="raw_symbol",
).to_df()

bars.to_csv("bars10.csv")


In [15]:
bars = pd.read_csv("bars10.csv")

In [31]:
bars.ask_sz_03.value_counts()

ask_sz_03
0      3610512
100       2726
246       1224
6          858
1          583
3          446
2          429
8          283
4          273
7          254
10         188
12         163
5          132
101         32
11          30
15          24
18          16
14          14
102         13
300          8
9            8
200          8
19           8
500          6
23           6
21           3
16           2
104          1
Name: count, dtype: int64

In [23]:
df = bars.copy()
df['ts_event'] = pd.to_datetime(df['ts_event'])
df = df.sort_values(['ts_event', 'sequence'])
df = df.set_index('ts_event')

depth_cols = []
for lvl in range(6):
    depth_cols += [
        f"bid_px_0{lvl}", f"ask_px_0{lvl}",
        f"bid_sz_0{lvl}", f"ask_sz_0{lvl}",
#        f"bid_ct_0{lvl}", f"ask_ct_0{lvl}",
    ]

orderbook_cols = depth_cols + ['symbol']
df_top5 = df[orderbook_cols]

df_top5_ff = df_top5.ffill()
df_per_sec = df_top5_ff.resample("1S").last().dropna(how="all")

print(df_per_sec.head())


/var/folders/l0/czv3z5cd6mn65b4g8mm5gsp00000gn/T/ipykernel_39973/3274078420.py:18: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_per_sec = df_top5_ff.resample("1S").last().dropna(how="all")


                           bid_px_00  ask_px_00  bid_sz_00  ask_sz_00  \
ts_event                                                                
2025-08-04 04:21:53+00:00        NaN        NaN        0.0        0.0   
2025-08-04 04:21:59+00:00        NaN        NaN        0.0        0.0   
2025-08-04 11:10:27+00:00        NaN        NaN        0.0        0.0   
2025-08-04 13:19:58+00:00      17.33       19.3      100.0      100.0   
2025-08-04 13:20:00+00:00      17.33       19.3        0.0        0.0   

                           bid_px_01  ask_px_01  bid_sz_01  ask_sz_01  \
ts_event                                                                
2025-08-04 04:21:53+00:00        NaN        NaN        0.0        0.0   
2025-08-04 04:21:59+00:00        NaN        NaN        0.0        0.0   
2025-08-04 11:10:27+00:00        NaN        NaN        0.0        0.0   
2025-08-04 13:19:58+00:00        NaN        NaN        0.0        0.0   
2025-08-04 13:20:00+00:00        NaN        NaN   

In [35]:
df_per_sec.dropna().ask_sz_00.value_counts()

ask_sz_00
100.0     34834
500.0     18531
300.0     18405
400.0     17871
600.0     17743
          ...  
3038.0        1
4338.0        1
2938.0        1
2838.0        1
2442.0        1
Name: count, Length: 2852, dtype: int64

In [ ]:

# --- Normalize timestamps & key columns ---
df = bars.copy()
ts_cols = [c for c in df.columns if c.lower() in ("ts_event", "ts_recv", "timestamp", "time", "ts")]
if not ts_cols:
    raise ValueError("No timestamp column found (expected one of ts_event, ts_recv, timestamp, time, ts).")
ts_col = ts_cols[0]
df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
df = df.dropna(subset=[ts_col])

if df[ts_col].dt.tz is None:
    df[ts_col] = df[ts_col].dt.tz_localize("UTC").dt.tz_convert("America/New_York")
else:
    df[ts_col] = df[ts_col].dt.tz_convert("America/New_York")

sym_cols = [c for c in df.columns if c.lower() in ("symbol", "sym", "ticker")]
if not sym_cols:
    raise ValueError("No symbol column found (expected one of symbol, sym, ticker).")
sym_col = sym_cols[0]

side_cols = [c for c in df.columns if c.lower() in ("side", "book_side")]
if not side_cols:
    raise ValueError("No side column found (expected one of side, book_side).")
side_col = side_cols[0]

price_cols = [c for c in df.columns if c.lower() in ("price", "px")]
size_cols  = [c for c in df.columns if c.lower() in ("size", "qty", "quantity", "sz")]
if not price_cols or not size_cols:
    raise ValueError("Missing price/size columns (expected price/px and size/qty/quantity/sz).")
price_col, size_col = price_cols[0], size_cols[0]

def normalize_side(x: str):
    if pd.isna(x): return np.nan
    s = str(x).strip().upper()
    if s in ("B", "BID", "BUY"): return "BID"
    if s in ("A", "ASK", "SELL", "OFFER", "OF"): return "ASK"
    return np.nan

df[side_col] = df[side_col].map(normalize_side)
df = df.dropna(subset=[side_col])

df = df[[ts_col, sym_col, side_col, price_col, size_col]].rename(
    columns={ts_col: "ts", sym_col: "symbol", side_col: "side", price_col: "price", size_col: "size"}
)

# --- Filter to RTH window ---
start = pd.Timestamp(start_time, tz="America/New_York")
end   = pd.Timestamp(end_time,   tz="America/New_York")

df = df[(df["ts"] >= start) & (df["ts"] <= end)]
t = df["ts"].dt
rth_mask = (t.hour > 9) | ((t.hour == 9) & (t.minute >= 30))
rth_mask &= (t.hour < 16) | ((t.hour == 16) & (t.minute == 0))
df = df[rth_mask].copy()

# Quote notional (for depth_notional + quote VWAP)
df["notional"] = df["price"] * df["size"]

# --- Binning params (30 minutes, start-labeled at 9:30) ---
# (9:30,10:00] -> 09:30 ; … ; (15:30,16:00] -> 15:30
bin_params = dict(
    rule="30T",
    label="left",
    closed="right",
    origin="start_day",
    offset="9h30min",
)

# --- (optional) enforcement knobs for spreads ---
ENFORCE_POSITIVE_SPREAD = True   # drop locked/crossed snapshots
MIN_TICK = None                  # set to 0.01 for US equities if you want ask >= bid + tick

# --- Helpers ---
def agg_best_price(group, side):
    if group.empty: return np.nan
    return group["price"].max() if side == "BID" else group["price"].min()

def agg_best_size(group, side):
    if group.empty: return np.nan
    best = group["price"].max() if side == "BID" else group["price"].min()
    return group.loc[group["price"] == best, "size"].sum()

def agg_vwap(group):
    if group.empty: return np.nan
    sz = group["size"].sum()
    if sz == 0: return np.nan
    return group["notional"].sum() / sz

def ohlc_from_series(s, name):
    rb = s.resample(**bin_params)
    return pd.concat([
        rb.first().rename(name + ".O"),
        rb.max().rename(  name + ".H"),
        rb.min().rename(  name + ".L"),
        rb.last().rename( name + ".C"),
    ], axis=1)

def compute_features_per_symbol(sym_df):
    df1 = sym_df.copy()

    # Separate & index
    bids = df1[df1["side"] == "BID"].set_index("ts").sort_index()
    asks = df1[df1["side"] == "ASK"].set_index("ts").sort_index()

    # --- Build synchronized top-of-book (collapse -> merge -> ffill together) ---
    bb_tick = bids.groupby(level=0)["price"].max().rename("bb")  # best bid per ts
    ba_tick = asks.groupby(level=0)["price"].min().rename("ba")  # best ask per ts

    top = pd.concat([bb_tick, ba_tick], axis=1).sort_index()
    top = top.ffill()  # carry both sides forward together

    # Enforce book consistency (no locked/crossed); optionally enforce a minimum tick
    if ENFORCE_POSITIVE_SPREAD:
        if MIN_TICK is None:
            # Drop any snapshots with ask <= bid, then ffill again to restore continuity
            bad = top["ba"] <= top["bb"]
            if bad.any():
                top.loc[bad, ["bb", "ba"]] = np.nan
                top = top.ffill()
        else:
            # Force ask >= bid + tick
            top["ba"] = np.maximum(top["ba"], top["bb"] + float(MIN_TICK))

    # Remove any leading rows still missing a side
    top = top.dropna(subset=["bb", "ba"])

    # Tick-level mid & spread
    mid_tick = ((top["bb"] + top["ba"]) / 2.0).rename("mid")
    spr_tick = (top["ba"] - top["bb"]).rename("spread")

    # --- OHLC from synchronized tick streams (start-labeled) ---
    md_ohlc = ohlc_from_series(mid_tick, "mid")
    sp_ohlc = ohlc_from_series(spr_tick, "spread")

    # Also compute start-labeled OHLC of the synchronized best bid/ask themselves
    bb_ohlc = ohlc_from_series(top["bb"], "best_price.BID")
    ba_ohlc = ohlc_from_series(top["ba"], "best_price.ASK")

    # Per-bin size, depth, and quote VWAPs (direct from raw updates)
    rb_b = bids.resample(**bin_params)
    rb_a = asks.resample(**bin_params)

    best_bid_sz = rb_b.apply(lambda g: agg_best_size(g, "BID")).rename("best_size.BID")
    best_ask_sz = rb_a.apply(lambda g: agg_best_size(g, "ASK")).rename("best_size.ASK")

    depth_bid_sz = rb_b["size"].sum().rename("depth_size.BID")
    depth_ask_sz = rb_a["size"].sum().rename("depth_size.ASK")
    depth_bid_nt = rb_b["notional"].sum().rename("depth_notional.BID")
    depth_ask_nt = rb_a["notional"].sum().rename("depth_notional.ASK")

    vwap_bid = rb_b.apply(agg_vwap).rename("vwap.BID")
    vwap_ask = rb_a.apply(agg_vwap).rename("vwap.ASK")

    # Imbalance & relative spread (use end-of-bin mid.C)
    depth_sum   = (depth_bid_sz + depth_ask_sz).replace(0, np.nan)
    depth_imbal = ((depth_bid_sz - depth_ask_sz) / depth_sum).rename("depth.imbalance")
    spread_rel  = ((ba_ohlc["best_price.ASK.C"] - bb_ohlc["best_price.BID.C"]) / md_ohlc["mid.C"]).rename("spread.rel")

    # Forward 30m return labeled at bin START
    log_ret_bin = (np.log(md_ohlc["mid.C"]) - np.log(md_ohlc["mid.O"])).rename("log_ret.bin")

    # after defining mid_tick
   # mid_10s = mid_tick.resample("1T").last().ffill()
    #rvol_mid = mid_10s.resample(**bin_params).var().rename("rvol_mid.var10s")

    # include in the final concat
    out = pd.concat(
        [
            bb_ohlc, ba_ohlc,
            best_bid_sz, best_ask_sz,
            depth_bid_sz, depth_ask_sz, depth_bid_nt, depth_ask_nt,
            vwap_bid, vwap_ask,
            md_ohlc, sp_ohlc,
            depth_imbal, spread_rel,
            md_ohlc["mid.O"].rename("mid.start"),
            md_ohlc["mid.C"].rename("mid.end"),
            log_ret_bin,
            #rvol_mid,
        ],
        axis=1,
    )


    # Restrict to RTH start grid 09:30..15:30 (no 16:00)
    out = out.loc[(out.index >= start) & (out.index <= end)]
    full_idx_start = pd.date_range(start=start, end=end, freq="30T", tz="America/New_York")
    full_idx_start = full_idx_start[(full_idx_start.time >= time(9, 30)) & (full_idx_start.time <= time(15, 30))]
    out = out.reindex(full_idx_start)

    return out

# --- Build features per symbol ---
features_list = []
for sym, g in df.groupby("symbol", sort=True):
    feats = compute_features_per_symbol(g)
    feats.columns = pd.MultiIndex.from_product([[sym], feats.columns])
    features_list.append(feats)

if not features_list:
    raise ValueError("No data after filtering. Check your inputs/timezone and raw feed.")

features = pd.concat(features_list, axis=1).sort_index()
features = features.groupby(features.index.date, group_keys=False).apply(lambda x: x.ffill())

# --- Column ordering (all start-labeled; no 16:00 row) ---
def sort_cols(cols):
    order = [
        "best_price.BID.O", "best_price.BID.H", "best_price.BID.L", "best_price.BID.C",
        "best_price.ASK.O", "best_price.ASK.H", "best_price.ASK.L", "best_price.ASK.C",

        "best_size.BID",  "best_size.ASK",

        "mid.O", "mid.H", "mid.L", "mid.C",
        "spread.O", "spread.H", "spread.L", "spread.C",

        "vwap.BID", "vwap.ASK",
        "depth_size.BID", "depth_size.ASK",
        "depth_notional.BID", "depth_notional.ASK",
        "depth.imbalance", "spread.rel",
        "mid.start", "mid.end", "log_ret.bin",
    ]
    def key(c):
        sym, feat = c
        try: k = order.index(feat)
        except ValueError: k = len(order)
        return (sym, k, feat)
    return sorted(cols, key=key)

features = features.reindex(columns=sort_cols(features.columns))

# --- Realized volatility over session using 30m forward returns (start-labeled) ---
rv = {}
for sym in symbols:
    col = (sym, "log_ret.bin")
    if col in features.columns:
        s = features[col].dropna()
        rv[sym] = float(np.sqrt(np.square(s).sum()))
session_realized_vol = pd.Series(rv, name="realized_vol_session").to_frame()

# --- Inspect ---
print(features.head(20))
print("\nPer-ticker session realized volatility (sqrt(sum of squared 30m forward log returns)):")
print(session_realized_vol)


In [150]:
features.loc[:, ("APA", ["spread.O", "spread.C", "spread.H", "spread.L"])]   # DataFrame


APA                           
                          spread.O spread.C spread.H spread.L
2025-08-02 09:30:00-04:00      NaN      NaN      NaN      NaN
2025-08-02 10:00:00-04:00      NaN      NaN      NaN      NaN
2025-08-02 10:30:00-04:00      NaN      NaN      NaN      NaN
2025-08-02 11:00:00-04:00      NaN      NaN      NaN      NaN
2025-08-02 11:30:00-04:00      NaN      NaN      NaN      NaN
...                            ...      ...      ...      ...
2025-09-02 13:30:00-04:00     0.05     0.01     1.08     0.01
2025-09-02 14:00:00-04:00     0.01     0.01     1.05     0.01
2025-09-02 14:30:00-04:00     0.01     0.01     1.11     0.01
2025-09-02 15:00:00-04:00     0.01     0.01     1.13     0.01
2025-09-02 15:30:00-04:00     0.01     0.01     1.13     0.01

[416 rows x 4 columns]

In [149]:
features

APA                                    \
                          best_price.BID.O best_price.BID.H best_price.BID.L   
2025-08-02 09:30:00-04:00              NaN              NaN              NaN   
2025-08-02 10:00:00-04:00              NaN              NaN              NaN   
2025-08-02 10:30:00-04:00              NaN              NaN              NaN   
2025-08-02 11:00:00-04:00              NaN              NaN              NaN   
2025-08-02 11:30:00-04:00              NaN              NaN              NaN   
...                                    ...              ...              ...   
2025-09-02 13:30:00-04:00            23.76            23.87            23.57   
2025-09-02 14:00:00-04:00            23.78            23.86            23.57   
2025-09-02 14:30:00-04:00            23.75            23.78            23.66   
2025-09-02 15:00:00-04:00            23.71            23.77            23.70   
2025-09-02 15:30:00-04:00            23.72            23.92            23.34   

                                                                              \
                          best_price.BID.C best_price.ASK.O best_price.ASK.H   
2025-08-02 09:30:00-04:00              NaN              NaN              NaN   
2025-08-02 10:00:00-04:00              NaN              NaN              NaN   
2025-08-02 10:30:00-04:00              NaN              NaN              NaN   
2025-08-02 11:00:00-04:00              NaN              NaN              NaN   
2025-08-02 11:30:00-04:00              NaN              NaN              NaN   
...                                    ...              ...              ...   
2025-09-02 13:30:00-04:00            23.78            23.81            24.88   
2025-09-02 14:00:00-04:00            23.75            23.79            24.88   
2025-09-02 14:30:00-04:00            23.71            23.76            24.88   
2025-09-02 15:00:00-04:00            23.72            23.72            24.88   
2025-09-02 15:30:00-04:00            23.86            23.73            24.88   

                                                                           \
                          best_price.ASK.L best_price.ASK.C best_size.BID   
2025-08-02 09:30:00-04:00              NaN              NaN           NaN   
2025-08-02 10:00:00-04:00              NaN              NaN           NaN   
2025-08-02 10:30:00-04:00              NaN              NaN           NaN   
2025-08-02 11:00:00-04:00              NaN              NaN           NaN   
2025-08-02 11:30:00-04:00              NaN              NaN           NaN   
...                                    ...              ...           ...   
2025-09-02 13:30:00-04:00            23.77            23.79        6800.0   
2025-09-02 14:00:00-04:00            23.74            23.76        2000.0   
2025-09-02 14:30:00-04:00            23.67            23.72         700.0   
2025-09-02 15:00:00-04:00            23.71            23.73        4200.0   
2025-09-02 15:30:00-04:00            23.71            23.87       13100.0   

                                         ...                            \
                          best_size.ASK  ...   vwap.ASK depth_size.BID   
2025-08-02 09:30:00-04:00           NaN  ...        NaN            NaN   
2025-08-02 10:00:00-04:00           NaN  ...        NaN            NaN   
2025-08-02 10:30:00-04:00           NaN  ...        NaN            NaN   
2025-08-02 11:00:00-04:00           NaN  ...        NaN            NaN   
2025-08-02 11:30:00-04:00           NaN  ...        NaN            NaN   
...                                 ...  ...        ...            ...   
2025-09-02 13:30:00-04:00         800.0  ...  23.820870      5242815.0   
2025-09-02 14:00:00-04:00       66259.0  ...  23.806780      6366553.0   
2025-09-02 14:30:00-04:00       17392.0  ...  23.723997      5008516.0   
2025-09-02 15:00:00-04:00       18245.0  ...  23.735494      5694184.0   
2025-09-02 15:30:00-04:00      147766.0  ...  23.784500     1238752

In [156]:

# --- Pull trades (primary) ----------------------------------------------------
trades_raw = client.timeseries.get_range(
    dataset=dataset,
    schema="trades",
    start=start_api,
    end=end_api,
    symbols=symbols,
    stype_in="raw_symbol",
).to_df()

# Ensure ts_event is datetime (looks like it already is)
ts = pd.to_datetime(trades_raw["ts_event"], errors="coerce")
if ts.dt.tz is None:                 
    ts = ts.dt.tz_localize("UTC")
trades_raw["ts_event"] = ts.dt.tz_convert("America/New_York")

trades_raw = trades_raw.set_index("ts_event").sort_index()

# Aggregations
agg_dict = {
    "price": ["first", "max", "min", "last", "mean", "median"],
    "size": ["sum", "mean", "max", "count"],
    "depth": ["mean", "max"],
    "ts_in_delta": ["mean", "max"]
}


# Resample to 30-min intervals
trades_resampled = trades_raw.resample("30T").agg(agg_dict)

# Flatten column names
trades_resampled.columns = ["_".join(col).strip() for col in trades_resampled.columns.values]

# Add VWAP
trades_resampled["vwap"] = (
    (trades_raw["price"] * trades_raw["size"]).resample("30T").sum()
    / trades_raw["size"].resample("30T").sum()
)

trades_resampled.drop(columns = ["depth_mean", "depth_max", "ts_in_delta_mean", "ts_in_delta_max"], inplace = True)
# Result: 30-min dataset with OHLC, volume, size stats, depth, latency, vwap
print(trades_resampled.head())



                           price_first  price_max  price_min  price_last  \
ts_event                                                                   
2025-08-04 09:30:00-04:00       18.390      18.55     18.275      18.290   
2025-08-04 10:00:00-04:00       18.280      18.50     18.240      18.465   
2025-08-04 10:30:00-04:00       18.455      18.60     18.380      18.535   
2025-08-04 11:00:00-04:00       18.530      18.57     18.450      18.510   
2025-08-04 11:30:00-04:00       18.520      18.53     18.420      18.425   

                           price_mean  price_median  size_sum   size_mean  \
ts_event                                                                    
2025-08-04 09:30:00-04:00   18.385254       18.3850     25250  106.991525   
2025-08-04 10:00:00-04:00   18.381000       18.4100     36787  102.186111   
2025-08-04 10:30:00-04:00   18.482149       18.4825     24226   73.859756   
2025-08-04 11:00:00-04:00   18.529910       18.5300     16223   97.143713   
2025-

/var/folders/l0/czv3z5cd6mn65b4g8mm5gsp00000gn/T/ipykernel_93530/1579536245.py:29: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  trades_resampled = trades_raw.resample("30T").agg(agg_dict)
/var/folders/l0/czv3z5cd6mn65b4g8mm5gsp00000gn/T/ipykernel_93530/1579536245.py:36: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  (trades_raw["price"] * trades_raw["size"]).resample("30T").sum()
/var/folders/l0/czv3z5cd6mn65b4g8mm5gsp00000gn/T/ipykernel_93530/1579536245.py:37: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  / trades_raw["size"].resample("30T").sum()


In [170]:
df

APA                                    \
                          best_price.BID.O best_price.BID.H best_price.BID.L   
2025-08-02 09:30:00-04:00              NaN              NaN              NaN   
2025-08-02 10:00:00-04:00              NaN              NaN              NaN   
2025-08-02 10:30:00-04:00              NaN              NaN              NaN   
2025-08-02 11:00:00-04:00              NaN              NaN              NaN   
2025-08-02 11:30:00-04:00              NaN              NaN              NaN   
...                                    ...              ...              ...   
2025-09-02 13:30:00-04:00            23.76            23.87            23.57   
2025-09-02 14:00:00-04:00            23.78            23.86            23.57   
2025-09-02 14:30:00-04:00            23.75            23.78            23.66   
2025-09-02 15:00:00-04:00            23.71            23.77            23.70   
2025-09-02 15:30:00-04:00            23.72            23.92            23.34   

                                                                              \
                          best_price.BID.C best_price.ASK.O best_price.ASK.H   
2025-08-02 09:30:00-04:00              NaN              NaN              NaN   
2025-08-02 10:00:00-04:00              NaN              NaN              NaN   
2025-08-02 10:30:00-04:00              NaN              NaN              NaN   
2025-08-02 11:00:00-04:00              NaN              NaN              NaN   
2025-08-02 11:30:00-04:00              NaN              NaN              NaN   
...                                    ...              ...              ...   
2025-09-02 13:30:00-04:00            23.78            23.81            24.88   
2025-09-02 14:00:00-04:00            23.75            23.79            24.88   
2025-09-02 14:30:00-04:00            23.71            23.76            24.88   
2025-09-02 15:00:00-04:00            23.72            23.72            24.88   
2025-09-02 15:30:00-04:00            23.86            23.73            24.88   

                                                                           \
                          best_price.ASK.L best_price.ASK.C best_size.BID   
2025-08-02 09:30:00-04:00              NaN              NaN           NaN   
2025-08-02 10:00:00-04:00              NaN              NaN           NaN   
2025-08-02 10:30:00-04:00              NaN              NaN           NaN   
2025-08-02 11:00:00-04:00              NaN              NaN           NaN   
2025-08-02 11:30:00-04:00              NaN              NaN           NaN   
...                                    ...              ...           ...   
2025-09-02 13:30:00-04:00            23.77            23.79        6800.0   
2025-09-02 14:00:00-04:00            23.74            23.76        2000.0   
2025-09-02 14:30:00-04:00            23.67            23.72         700.0   
2025-09-02 15:00:00-04:00            23.71            23.73        4200.0   
2025-09-02 15:30:00-04:00            23.71            23.87       13100.0   

                                         ...                            \
                          best_size.ASK  ... tr_price_max tr_price_min   
2025-08-02 09:30:00-04:00           NaN  ...          NaN          NaN   
2025-08-02 10:00:00-04:00           NaN  ...          NaN          NaN   
2025-08-02 10:30:00-04:00           NaN  ...          NaN          NaN   
2025-08-02 11:00:00-04:00           NaN  ...          NaN          NaN   
2025-08-02 11:30:00-04:00           NaN  ...          NaN          NaN   
...                                 ...  ...          ...          ...   
2025-09-02 13:30:00-04:00         800.0  ...        23.87        23.77   
2025-09-02 14:00:00-04:00       66259.0  ...        23.86        23.73   
2025-09-02 14:30:00-04:00       17392.0  ...        23.78        23.66   
2025-09-02 15:00:00-04:00       18245.0  ...        23.77        23.70   
2025-09-02 15:30:00-04:00      147766.0  ...        23.87        23

In [191]:
sym = symbols[0] if isinstance(symbols, (list, tuple, pd.Index)) else symbols
tr = trades_resampled.copy()
tr.columns = pd.MultiIndex.from_product([[sym], [f"tr_{c}" for c in tr.columns]])
df = features.join(tr, how="left")
def add_vol(df):
    if not isinstance(df.columns, pd.MultiIndex):
        raise ValueError("Expected MultiIndex columns: (ticker, field)")

    tickers = df.columns.get_level_values(0).unique()

    for sym in tickers:
        if (sym, "log_ret.bin") in df.columns:
            df[(sym, "vol")] = df[(sym, "log_ret.bin")] ** 2
            df[(sym, "log_ret")] = df[(sym, "log_ret.bin")]
            df.drop(columns = (sym, "log_ret.bin"), inplace = True)
    return df

# usage
df = add_vol(df)

df.dropna(inplace = True)

In [192]:
df

APA                                    \
                          best_price.BID.O best_price.BID.H best_price.BID.L   
2025-08-04 09:30:00-04:00            16.68            18.57            16.68   
2025-08-04 10:00:00-04:00            18.22            18.51            18.17   
2025-08-04 10:30:00-04:00            18.46            18.61            18.20   
2025-08-04 11:00:00-04:00            18.54            18.58            18.48   
2025-08-04 11:30:00-04:00            18.52            18.54            18.42   
...                                    ...              ...              ...   
2025-09-02 13:30:00-04:00            23.76            23.87            23.57   
2025-09-02 14:00:00-04:00            23.78            23.86            23.57   
2025-09-02 14:30:00-04:00            23.75            23.78            23.66   
2025-09-02 15:00:00-04:00            23.71            23.77            23.70   
2025-09-02 15:30:00-04:00            23.72            23.92            23.34   

                                                                              \
                          best_price.BID.C best_price.ASK.O best_price.ASK.H   
2025-08-04 09:30:00-04:00            18.22            19.90            19.90   
2025-08-04 10:00:00-04:00            18.46            18.28            19.90   
2025-08-04 10:30:00-04:00            18.54            18.47            19.31   
2025-08-04 11:00:00-04:00            18.52            18.55            19.31   
2025-08-04 11:30:00-04:00            18.42            19.31            19.31   
...                                    ...              ...              ...   
2025-09-02 13:30:00-04:00            23.78            23.81            24.88   
2025-09-02 14:00:00-04:00            23.75            23.79            24.88   
2025-09-02 14:30:00-04:00            23.71            23.76            24.88   
2025-09-02 15:00:00-04:00            23.72            23.72            24.88   
2025-09-02 15:30:00-04:00            23.86            23.73            24.88   

                                                                           \
                          best_price.ASK.L best_price.ASK.C best_size.BID   
2025-08-04 09:30:00-04:00            18.27            18.28        2900.0   
2025-08-04 10:00:00-04:00            18.25            18.47         200.0   
2025-08-04 10:30:00-04:00            18.39            18.55       19300.0   
2025-08-04 11:00:00-04:00            18.49            19.31       18900.0   
2025-08-04 11:30:00-04:00            18.43            18.44        2500.0   
...                                    ...              ...           ...   
2025-09-02 13:30:00-04:00            23.77            23.79        6800.0   
2025-09-02 14:00:00-04:00            23.74            23.76        2000.0   
2025-09-02 14:30:00-04:00            23.67            23.72         700.0   
2025-09-02 15:00:00-04:00            23.71            23.73        4200.0   
2025-09-02 15:30:00-04:00            23.71            23.87       13100.0   

                                         ...                              \
                          best_size.ASK  ... tr_price_last tr_price_mean   
2025-08-04 09:30:00-04:00         300.0  ...        18.290     18.385254   
2025-08-04 10:00:00-04:00        6200.0  ...        18.465     18.381000   
2025-08-04 10:30:00-04:00        4050.0  ...        18.535     18.482149   
2025-08-04 11:00:00-04:00        2800.0  ...        18.510     18.529910   
2025-08-04 11:30:00-04:00       10950.0  ...        18.425     18.477638   
...                                 ...  ...           ...           ...   
2025-09-02 13:30:00-04:00         800.0  ...        23.785     23.816762   
2025-09-02 14:00:00-04:00       66259.0  ...        23.745     23.805263   
2025-09-02 14:30:00-04:00       17392.0  ...        23.710     23.714470   
2025-09-02 15:00:00-04:00       18245.0  ...        23.720     23.728401   
2025-09-02 15:30:00-04:00      147766.0  ..

In [193]:
#df.to_csv("df_apa2.csv", index=True)